In [1]:
# loading the liberaries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import pickle
import joblib

In [2]:
# loading the dataset
df = pd.read_csv("Car Price.csv")
df

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,seats
0,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,5.0
1,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,5.0
2,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,5.0
3,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,5.0
4,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,5.0
...,...,...,...,...,...,...,...,...,...,...
8123,2013,320000,110000,Petrol,Individual,Manual,First Owner,18.50,1197.0,5.0
8124,2007,135000,119000,Diesel,Individual,Manual,Fourth & Above Owner,16.80,1493.0,5.0
8125,2009,382000,120000,Diesel,Individual,Manual,First Owner,19.30,1248.0,5.0
8126,2013,290000,25000,Diesel,Individual,Manual,First Owner,23.57,1396.0,5.0


In [3]:
# checking for null values
df.isnull().sum()

year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
seats            221
dtype: int64

In [4]:
# filling the null values with random values 
df["mileage"]=df["mileage"].fillna(20) # filling the null values in the 'mileage' column with 20
df["engine"]=df["engine"].fillna(1500) # filling the null values in the 'engine' column with 1500
df["seats"]=df["seats"].fillna(5) # filling the null values in the 'seats' column with 5

In [5]:
# handling categorical data 
label=LabelEncoder() # creating an instance of the LabelEncoder class

df["fuel_encoded"]=label.fit_transform(df["fuel"]) 
var=dict(zip(df["fuel"],df["fuel_encoded"]))
print(var) # printing the dictionary

df["seller_type_encoded"]=label.fit_transform(df["seller_type"]) # encoding the 'seller_type' column
var1=dict(zip(df["seller_type"],df["seller_type_encoded"])) # creating a dictionary of the 'seller_type' column and the encoded 'seller_type' column
print(var1) # printing the dictionary

df["transmission_encoded"]=label.fit_transform(df["transmission"]) # encoding the 'transmission' column
var2=dict(zip(df["transmission"],df["transmission_encoded"])) # creating a dictionary
print(var2) # printing the dictionary

df["owner_encoded"]=label.fit_transform(df["owner"]) # encoding the 'owner' column
var3=dict(zip(df["owner"],df["owner_encoded"])) # creating a dictionary of the
print(var3) # printing the dictionary

{'Diesel': 1, 'Petrol': 3, 'LPG': 2, 'CNG': 0}
{'Individual': 1, 'Dealer': 0, 'Trustmark Dealer': 2}
{'Manual': 1, 'Automatic': 0}
{'First Owner': 0, 'Second Owner': 2, 'Third Owner': 4, 'Fourth & Above Owner': 1, 'Test Drive Car': 3}


In [6]:
# drop orignal columns
df.drop(["fuel","seller_type","transmission","owner"],axis=1,inplace=True) # dropping the original columns

In [7]:
# feature selection 
x=df.drop("selling_price",axis=1) # dropping the target variable from the dataset
#x=df.drop(columns=["selling_price"]) # dropping the 'name' and 'year' columns from the dataset
y=df["selling_price"] # target variable

In [8]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42) # splitting the dataset into training and testing sets

In [9]:
scaler=StandardScaler() # creating an instance of the StandardScaler class
x_train_scaled=scaler.fit_transform(x_train) # scaling the training data
x_test_scaled=scaler.transform(x_test) # scaling the testing data   

In [10]:
# apply pca
pca=PCA(n_components=0.95) # creating an instance of the PCA class
x_train_pca=pca.fit_transform(x_train_scaled) # applying PCA on the training data
x_test_pca=pca.transform(x_test_scaled) # applying PCA on the testing data

In [11]:
# Hyperparameter tuning
param_grid = {"n_estimators": np.arange(50, 301, 50), "max_depth": np.arange(1,20,2)} # creating a dictionary of hyperparameters

In [12]:
model=RandomForestRegressor(random_state=42) # creating an instance of the RandomForestRegressor class

In [13]:
grid_search=GridSearchCV(estimator=model,
            param_grid=param_grid,
            scoring='r2',
            cv=5,
            n_jobs=-1,
            verbose=1) # creating an instance of the GridSearchCV class

#model tuning on pca data
grid_search.fit(x_train_pca,y_train) # fitting the model on the training data

Fitting 5 folds for each of 60 candidates, totalling 300 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': array([ 1, 3..., 15, 17, 19]), 'n_estimators': array([ 50, 1...00, 250, 300])}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(100)


In [14]:
y_pred=grid_search.predict(x_test_pca) # predicting the target variable on the testing data
r2_score(y_test, y_pred) # calculating the r2 score


0.933188521235204

In [15]:
y_pred=grid_search.predict(x_train_pca) # predicting the target variable on the testing data
r2_score(y_train, y_pred) # calculating the r2 score


0.9882578703989541

In [16]:
# User Input for Car Price Prediction

print("\nEnter Car Details:")

year = int(input("Enter car manufacturing year: "))
km_driven = float(input("Enter kilometers driven: "))

print("\nFuel Type:")
print("0 = CNG")
print("1 = Diesel")
print("2 = LPG")
print("3 = Petrol")
fuel_encoded = int(input("Enter fuel type code: "))

print("\nSeller Type:")
print("0 = Dealer")
print("1 = Individual")
print("2 = Trustmark Dealer")
seller_type_encoded = int(input("Enter seller type code: "))

print("\nTransmission:")
print("0 = Automatic")
print("1 = Manual")
transmission_encoded = int(input("Enter transmission code: "))

print("\nOwner:")
print("0 = First Owner")
print("1 = Fourth & Above Owner")
print("2 = Second Owner")
print("3 = Test Drive Car")
print("4 = Third Owner")
owner_encoded = int(input("Enter owner code: "))

mileage = float(input("Enter mileage: "))
engine = float(input("Enter engine capacity: "))
seats = float(input("Enter number of seats: "))


# Creating input array
user_data = np.array([[
    year,
    km_driven,
    fuel_encoded,
    seller_type_encoded,
    transmission_encoded,
    owner_encoded,
    mileage,
    engine,
    seats
]])


# Step 1: Scale the data
user_data_scaled = scaler.transform(user_data)


# Step 2: Apply PCA
user_data_pca = pca.transform(user_data_scaled)


# Step 3: Predict car price
prediction = grid_search.predict(user_data_pca)


print("\n-----------------------------")
print("Predicted Car Price:", prediction[0])
print("-----------------------------")


Enter Car Details:

Fuel Type:
0 = CNG
1 = Diesel
2 = LPG
3 = Petrol

Seller Type:
0 = Dealer
1 = Individual
2 = Trustmark Dealer

Transmission:
0 = Automatic
1 = Manual

Owner:
0 = First Owner
1 = Fourth & Above Owner
2 = Second Owner
3 = Test Drive Car
4 = Third Owner

-----------------------------
Predicted Car Price: 174289.96
-----------------------------


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
